In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

print("Train shape:", train.shape)
print("Test shape:",  test.shape)
print("\nTarget distribution:")
print(train['Transported'].value_counts())

Train shape: (8693, 14)
Test shape: (4277, 13)

Target distribution:
Transported
True     4378
False    4315
Name: count, dtype: int64


In [3]:
# ── SAVE TARGET + COMBINE ────────────────────
ntrain  = train.shape[0]
y_train = train['Transported'].astype(int)

all_data = pd.concat([
    train.drop('Transported', axis=1),
    test
]).reset_index(drop=True)

print(f"Combined shape: {all_data.shape}")

Combined shape: (12970, 13)


In [4]:
# ── EXTRACT BASIC FEATURES ───────────────────

# Cabin - Deck, CabinNum, Side
all_data[['Deck','CabinNum','Side']] = all_data['Cabin'].str.split(
    '/', expand=True)
all_data['CabinNum'] = pd.to_numeric(
    all_data['CabinNum'], errors='coerce')

# Group features from PassengerId
all_data['Group'] = all_data['PassengerId'].str.split('_').str[0]
all_data['GroupSize'] = all_data.groupby('Group')['Group'].transform('count')
all_data['IsAlone'] = (all_data['GroupSize'] == 1).astype(int)

# Spending
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
all_data['TotalSpend'] = all_data[spend_cols].sum(axis=1)
all_data['ZeroSpend']  = (all_data['TotalSpend'] == 0).astype(int)

print("Basic features done")
print(f"Shape: {all_data.shape}")

Basic features done
Shape: (12970, 21)


In [5]:
# ── MISSING VALUE HANDLING ─────────────

# CryoSleep=True → spending must be 0
for col in spend_cols:
    cryo_true  = all_data['CryoSleep'] == 'True'
    cryo_false = all_data['CryoSleep'] == 'False'
    
    all_data.loc[cryo_true, col]  = all_data.loc[cryo_true, col].fillna(0)
    
    median_val = all_data.loc[cryo_false, col].median()
    all_data.loc[cryo_false, col] = all_data.loc[cryo_false, col].fillna(median_val)
    
    all_data[col] = all_data[col].fillna(all_data[col].median())

# Recalculate after filling
all_data['TotalSpend'] = all_data[spend_cols].sum(axis=1)
all_data['ZeroSpend']  = (all_data['TotalSpend'] == 0).astype(int)

# Fill CryoSleep using ZeroSpend hint
all_data['CryoSleep'] = all_data['CryoSleep'].fillna(
    all_data['ZeroSpend'].map({1: 'True', 0: 'False'})
)

# Fill categoricals with mode
mode_cols = ['HomePlanet','Destination','VIP','Deck','Side']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# Fill numerics with median
all_data['Age']      = all_data['Age'].fillna(all_data['Age'].median())
all_data['CabinNum'] = all_data['CabinNum'].fillna(all_data['CabinNum'].median())

print(f"Missing after basic fill: {all_data.isnull().sum().sum()}")

# ── DROP UNUSED COLUMNS ─────────────────────
# Cabin - already extracted Deck, CabinNum, Side
# Name  - no useful signal
all_data = all_data.drop(columns=['Cabin', 'Name'], errors='ignore')

# Verify
print(f"Missing remaining: {all_data.isnull().sum().sum()}")
print(f"Shape: {all_data.shape}")

Missing after basic fill: 593
Missing remaining: 0
Shape: (12970, 19)


/tmp/ipykernel_16/2866332676.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_data[col] = all_data[col].fillna(all_data[col].mode()[0])


In [6]:
# ── FEATURE ENGINEERING ──────────────────────

# Log transform spending
for col in spend_cols:
    all_data[f'Log_{col}'] = np.log1p(all_data[col])

# Zero spend per category
for col in spend_cols:
    all_data[f'{col}_Zero'] = (all_data[col] == 0).astype(int)

# Number of zero spending categories
all_data['NumZeroSpend'] = all_data[
    [f'{c}_Zero' for c in spend_cols]].sum(axis=1)

# Age features
all_data['IsChild'] = (all_data['Age'] < 18).astype(int)
all_data['AgeBin']  = pd.cut(
    all_data['Age'],
    bins=[0, 12, 18, 25, 35, 50, 65, 100],
    labels=[0, 1, 2, 3, 4, 5, 6]
).astype(float)

# CryoSleep alone
all_data['CryoAlone'] = (
    (all_data['CryoSleep'] == 'True') &
    (all_data['GroupSize'] == 1)
).astype(int)

# Europa + CryoSleep interaction
all_data['Europa_Cryo'] = (
    (all_data['HomePlanet'] == 'Europa') &
    (all_data['CryoSleep'] == 'True')
).astype(int)

# Deck risk group
all_data['DeckGroup'] = all_data['Deck'].map({
    'A': 1, 'B': 3, 'C': 3,
    'D': 1, 'E': 0, 'F': 1,
    'G': 2, 'T': 0
})

# Group transport rate — do groups get transported together?
group_transport = train.copy()
group_transport['Group'] = train['PassengerId'].str.split('_').str[0]
group_rate = group_transport.groupby('Group')['Transported'].mean()

all_data['GroupTransportRate'] = all_data['Group'].map(group_rate)
all_data['GroupTransportRate'] = all_data['GroupTransportRate'].fillna(0.5)

# Big spender flag
all_data['BigSpender'] = (all_data['TotalSpend'] > 2000).astype(int)

print("Feature engineering done")
print(f"Shape: {all_data.shape}")

Feature engineering done
Shape: (12970, 37)


In [7]:
# ── DROP USELESS + ENCODE ────────────────────

# Drop columns we don't need
drop_cols = ['PassengerId', 'Group', 'Cabin', 'Name']
all_data  = all_data.drop(
    columns=[c for c in drop_cols if c in all_data.columns],
    errors='ignore'
)

all_data = all_data.drop(columns=['GroupTransportRate'], errors='ignore')

# Remove duplicate columns BEFORE encoding
all_data = all_data.loc[:, ~all_data.columns.duplicated()]

# One-hot encode categoricals
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
print(f"Encoding {len(cat_cols)} columns: {cat_cols}")
all_data = pd.get_dummies(all_data, columns=cat_cols)

# Remove duplicates AFTER encoding too
all_data = all_data.loc[:, ~all_data.columns.duplicated()]

# Fix bool columns safely
bool_mask = all_data.dtypes == bool
bool_cols = all_data.columns[bool_mask].tolist()
print(f"Bool columns to fix: {len(bool_cols)}")
for col in bool_cols:
    all_data[col] = all_data[col].astype(int)

# Final dedup
all_data = all_data.loc[:, ~all_data.columns.duplicated()]

# Fix AgeBin
all_data['AgeBin'] = all_data['AgeBin'].fillna(all_data['AgeBin'].median())

# Verify
print(f"\nFinal shape: {all_data.shape}")
print(f"Missing: {all_data.isnull().sum().sum()}")
print(f"Dtypes:\n{all_data.dtypes.value_counts()}")

Encoding 5 columns: ['HomePlanet', 'CryoSleep', 'Destination', 'Deck', 'Side']
Bool columns to fix: 19

Final shape: (12970, 47)
Missing: 0
Dtypes:
int64      33
float64    14
Name: count, dtype: int64


In [8]:
# ── SPLIT BACK ───────────────────────────────
X_train = all_data[:ntrain].copy()
X_test  = all_data[ntrain:].copy()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")

X_train: (8693, 47)
X_test:  (4277, 47)
y_train: (8693,)


In [9]:
# ── TRAIN MODELS ─────────────────────────────
models = {
    'XGBoost': XGBClassifier(
        n_estimators=2000,
        learning_rate=0.01,
        max_depth=5,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.01,
        max_depth=5,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
}

results = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='accuracy'
    )
    acc = scores.mean()
    std = scores.std()
    results[name] = acc
    print(f"{name:15} Accuracy: {acc:.4f} (+/- {std:.4f})")

print(f"\nBest model: {max(results, key=results.get)}")

XGBoost         Accuracy: 0.7839 (+/- 0.0250)
LightGBM        Accuracy: 0.7845 (+/- 0.0268)
RandomForest    Accuracy: 0.7992 (+/- 0.0162)

Best model: RandomForest


In [10]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained")

# Probabilities
prob_xgb  = models['XGBoost'].predict_proba(X_test)[:,1]
prob_lgbm = models['LightGBM'].predict_proba(X_test)[:,1]
prob_rf   = models['RandomForest'].predict_proba(X_test)[:,1]

# Weighted ensemble
ensemble_prob = (
    prob_xgb  * 0.35 +
    prob_lgbm * 0.35 +
    prob_rf   * 0.30
)

# Convert to True/False
predictions = (ensemble_prob > 0.5).astype(bool)

# Submit
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': predictions
})
output.to_csv('submission.csv', index=False)
print(output.head(10))
print(f"\nTransported True:  {predictions.sum()}")
print(f"Transported False: {(~predictions).sum()}")
print("Done!")

XGBoost trained
LightGBM trained
RandomForest trained
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
5     0027_01         True
6     0029_01         True
7     0032_01         True
8     0032_02         True
9     0033_01         True

Transported True:  2196
Transported False: 2081
Done!
